In [ ]:
import math

from assignment_3_q_1 import (
    loadConll2003Dataset,
    preprocessDatasetForGlove,
    loadGloveWikiGigaword100,
    SOS_TOKEN,
    EOS_TOKEN,
    UNK_TOKEN,
)

In [ ]:
def stripBound(example):
    """Drop SOS/EOS tokens and matching boundary tags if present."""
    tokens, tags = example["tokens"], example["ner_tags"]
    if tokens and tokens[0] == SOS_TOKEN and tokens[-1] == EOS_TOKEN:
        return tokens[1:-1], tags[1:-1]
    return tokens, tags


def trainHmm(train_examples, label_names, smoothing=1.0):
    """First-order HMM with add-`smoothing` (Laplace). Returns plain probs pi, A, B (not logs)."""
    alpha = smoothing
    n_tags = len(label_names)

    stripped = [stripBound(ex) for ex in train_examples]
    stripped = [(t, y) for t, y in stripped if t]

    vocab = {UNK_TOKEN}
    for tokens, _ in stripped:
        vocab.update(tokens)
    vocab_list = sorted(vocab)
    vocab_size = len(vocab_list)
    word_to_ix = {w: i for i, w in enumerate(vocab_list)}

    pi_counts = [alpha] * n_tags
    trans_counts = [[alpha] * n_tags for _ in range(n_tags)]
    emit_counts = [[alpha] * vocab_size for _ in range(n_tags)]

    for tokens, tags in stripped:
        pi_counts[tags[0]] += 1
        for y, w in zip(tags, tokens):
            emit_counts[y][word_to_ix[w]] += 1
        for y_prev, y_next in zip(tags, tags[1:]):
            trans_counts[y_prev][y_next] += 1

    pi_den = sum(pi_counts)
    pi = [c / pi_den for c in pi_counts]

    A = []
    for row in trans_counts:
        den = sum(row)
        A.append([c / den for c in row])

    B = []
    for row in emit_counts:
        den = sum(row)
        B.append([c / den for c in row])

    return {
        "label_names": label_names,
        "n_tags": n_tags,
        "pi": pi,
        "A": A,
        "B": B,
        "word_to_ix": word_to_ix,
        "vocab_size": vocab_size,
    }

In [ ]:
def viterbiDecode(words, model):
    """Best tag sequence; converts pi/A/B to logs inside (avoids underflow)."""
    if not words:
        return []

    n_tags = model["n_tags"]
    pi = model["pi"]
    A = model["A"]
    B = model["B"]
    word_to_ix = model["word_to_ix"]
    unk_ix = word_to_ix[UNK_TOKEN]

    log_pi = [math.log(p) for p in pi]
    log_A = [[math.log(a) for a in row] for row in A]
    log_B = [[math.log(b) for b in row] for row in B]

    obs_ix = [word_to_ix[w] if w in word_to_ix else unk_ix for w in words]
    T = len(obs_ix)

    dp = [[-math.inf] * n_tags for _ in range(T)]
    back = [[0] * n_tags for _ in range(T)]

    for s in range(n_tags):
        dp[0][s] = log_pi[s] + log_B[s][obs_ix[0]]

    for t in range(1, T):
        for s in range(n_tags):
            best_val = -math.inf
            best_prev = 0
            emit = log_B[s][obs_ix[t]]
            for sp in range(n_tags):
                cand = dp[t - 1][sp] + log_A[sp][s] + emit
                if cand > best_val:
                    best_val = cand
                    best_prev = sp
            dp[t][s] = best_val
            back[t][s] = best_prev

    best_last = max(range(n_tags), key=lambda s: dp[T - 1][s])
    path = [0] * T
    path[T - 1] = best_last
    for t in range(T - 2, -1, -1):
        path[t] = back[t + 1][path[t + 1]]
    return path


def evaluateHmmTagger(model, examples):
    total_correct = 0
    total_tokens = 0
    for ex in examples:
        tokens, gold = stripBound(ex)
        if not tokens:
            continue
        pred = viterbiDecode(tokens, model)
        total_correct += sum(int(p == g) for p, g in zip(pred, gold))
        total_tokens += len(gold)
    return total_correct / total_tokens if total_tokens else 0.0

In [ ]:
# Train / decode (GloVe preprocessing + HMM)
dataset_dir = "data/conll2003"
dataset = loadConll2003Dataset(dataset_dir)
embedding = loadGloveWikiGigaword100()
preprocessed = preprocessDatasetForGlove(dataset, embedding)

hmm = trainHmm(preprocessed["train"], preprocessed["label_names"])
print("validation token accuracy:", evaluateHmmTagger(hmm, preprocessed["validation"]))
print("test token accuracy:", evaluateHmmTagger(hmm, preprocessed["test"]))